# Credit Card Fraud Detection
### Project 6 — AI: From Basics to Agentic AI (Summer Internship)
**Technology:** Machine Learning (Isolation Forest)
**Source of Data:** Kaggle — Credit Card Fraud Detection dataset

**Objective:** Detect fraudulent credit card transactions in a highly imbalanced dataset using
unsupervised anomaly detection (Isolation Forest), and benchmark it against a supervised model
(Random Forest) to understand the trade-offs of each approach.

**Project Outcomes:**
1. Anomaly detection on real-world imbalanced financial data
2. Precision/recall improvement through model comparison and tuning
3. Risk analysis — understanding which transactions get flagged and why


## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score,
                              roc_curve, precision_recall_curve, average_precision_score)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100


## 2. Load the Dataset

The dataset contains transactions made by European cardholders in September 2013.
It has 284,807 transactions, of which only 492 (0.17%) are fraudulent — a classic
severely imbalanced classification problem.

Features `V1`–`V28` are the result of a PCA transformation (the original features are
confidential for privacy reasons). `Time` and `Amount` are the only features not transformed.

In [ ]:
df = pd.read_csv("data/creditcard.csv")
print("Shape:", df.shape)
df.head()

## 3. Exploratory Data Analysis

In [ ]:
print(df["Class"].value_counts())
print()
print(df["Class"].value_counts(normalize=True) * 100)

In [ ]:
fig, ax = plt.subplots(figsize=(5,4))
sns.countplot(x="Class", data=df, hue="Class", palette=["#4C72B0","#C44E52"], legend=False, ax=ax)
ax.set_xticks([0,1]); ax.set_xticklabels(["Normal","Fraud"])
ax.set_yscale("log")
ax.set_title("Class Distribution (log scale)")
plt.show()

In [ ]:
print(df.isnull().sum().sum(), "missing values")
print(df.duplicated().sum(), "duplicate rows")
df["Amount"].describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11,4))
sns.histplot(df[df.Class==0]["Amount"], bins=50, ax=axes[0], color="#4C72B0")
axes[0].set_title("Amount - Normal transactions"); axes[0].set_xlim(0,2000)
sns.histplot(df[df.Class==1]["Amount"], bins=50, ax=axes[1], color="#C44E52")
axes[1].set_title("Amount - Fraudulent transactions"); axes[1].set_xlim(0,2000)
plt.tight_layout(); plt.show()

In [ ]:
corr = df.corr()["Class"].drop("Class").sort_values()
plt.figure(figsize=(6,8))
corr.plot(kind="barh", color=np.where(corr>0, "#C44E52","#4C72B0"))
plt.title("Correlation of features with Class (fraud)")
plt.tight_layout(); plt.show()

**Observations:**
- The dataset is extremely imbalanced (0.17% fraud) — accuracy alone would be a misleading metric.
- Fraudulent transactions tend to cluster at lower amounts but include some high-value outliers.
- Several PCA components (e.g. V17, V14, V12, V10) show noticeably stronger correlation with fraud than others.

## 4. Preprocessing

`Amount` and `Time` are on very different scales than the PCA components, so we scale them with
`RobustScaler` (less sensitive to the extreme outliers fraud data tends to have).

In [ ]:
data = df.drop_duplicates().reset_index(drop=True).copy()

scaler = RobustScaler()
data["scaled_amount"] = scaler.fit_transform(data[["Amount"]])
data["scaled_time"] = scaler.fit_transform(data[["Time"]])
data = data.drop(["Amount","Time"], axis=1)

X = data.drop("Class", axis=1)
y = data["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)
print("Train:", X_train.shape, " Test:", X_test.shape)
print("Fraud rate - train: %.4f%%  test: %.4f%%" % (y_train.mean()*100, y_test.mean()*100))

## 5. Model 1 — Isolation Forest (Unsupervised)

Isolation Forest detects anomalies by randomly partitioning the data. Because anomalies are
"few and different," they tend to get isolated in fewer random splits than normal points —
their average path length across the trees is shorter, which is used as the anomaly score.

This is the core technique specified for this project. Note it is trained **without** the
labels (`y_train` is only used to estimate the expected contamination rate).

In [ ]:
contamination = y_train.mean()

iso_forest = IsolationForest(
    n_estimators=200,
    contamination=contamination,
    random_state=42,
    n_jobs=-1
)
iso_forest.fit(X_train)

# Isolation Forest outputs +1 (normal) / -1 (anomaly) -> convert to 1=fraud, 0=normal
iso_pred = (iso_forest.predict(X_test) == -1).astype(int)
iso_score = -iso_forest.score_samples(X_test)  # higher = more anomalous

print(classification_report(y_test, iso_pred, target_names=["Normal","Fraud"], digits=4))

In [ ]:
cm = confusion_matrix(y_test, iso_pred)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Normal","Fraud"], yticklabels=["Normal","Fraud"])
plt.title("Confusion Matrix - Isolation Forest")
plt.ylabel("Actual"); plt.xlabel("Predicted")
plt.tight_layout(); plt.show()

## 6. Model 2 — Random Forest (Supervised Baseline)

Included as a benchmark: since we *do* have labels in this dataset, it's useful to see how much
better a supervised model does, to put the Isolation Forest's unsupervised performance in context.
`class_weight="balanced"` compensates for the imbalance without needing to oversample.

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)
rf_score = rf.predict_proba(X_test)[:,1]

print(classification_report(y_test, rf_pred, target_names=["Normal","Fraud"], digits=4))

In [ ]:
cm = confusion_matrix(y_test, rf_pred)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Normal","Fraud"], yticklabels=["Normal","Fraud"])
plt.title("Confusion Matrix - Random Forest")
plt.ylabel("Actual"); plt.xlabel("Predicted")
plt.tight_layout(); plt.show()

In [ ]:
importances = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False).head(10)
plt.figure(figsize=(6,5))
importances.sort_values().plot(kind="barh", color="#55A868")
plt.title("Top 10 Feature Importances - Random Forest")
plt.tight_layout(); plt.show()

## 7. Model Comparison

In [ ]:
fpr_i, tpr_i, _ = roc_curve(y_test, iso_score)
fpr_r, tpr_r, _ = roc_curve(y_test, rf_score)

plt.figure(figsize=(6,5))
plt.plot(fpr_i, tpr_i, label=f"Isolation Forest (AUC={roc_auc_score(y_test, iso_score):.3f})")
plt.plot(fpr_r, tpr_r, label=f"Random Forest (AUC={roc_auc_score(y_test, rf_score):.3f})")
plt.plot([0,1],[0,1],"k--",alpha=0.4)
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison"); plt.legend()
plt.tight_layout(); plt.show()

In [ ]:
prec_i, rec_i, _ = precision_recall_curve(y_test, iso_score)
prec_r, rec_r, _ = precision_recall_curve(y_test, rf_score)

plt.figure(figsize=(6,5))
plt.plot(rec_i, prec_i, label=f"Isolation Forest (AP={average_precision_score(y_test, iso_score):.3f})")
plt.plot(rec_r, prec_r, label=f"Random Forest (AP={average_precision_score(y_test, rf_score):.3f})")
plt.xlabel("Recall"); plt.ylabel("Precision")
plt.title("Precision-Recall Curve Comparison"); plt.legend()
plt.tight_layout(); plt.show()

In [ ]:
summary = pd.DataFrame([
    {
        "Model": "Isolation Forest",
        "Precision (Fraud)": classification_report(y_test, iso_pred, output_dict=True)["1"]["precision"],
        "Recall (Fraud)": classification_report(y_test, iso_pred, output_dict=True)["1"]["recall"],
        "F1 (Fraud)": classification_report(y_test, iso_pred, output_dict=True)["1"]["f1-score"],
        "ROC-AUC": roc_auc_score(y_test, iso_score),
        "PR-AUC": average_precision_score(y_test, iso_score),
    },
    {
        "Model": "Random Forest",
        "Precision (Fraud)": classification_report(y_test, rf_pred, output_dict=True)["1"]["precision"],
        "Recall (Fraud)": classification_report(y_test, rf_pred, output_dict=True)["1"]["recall"],
        "F1 (Fraud)": classification_report(y_test, rf_pred, output_dict=True)["1"]["f1-score"],
        "ROC-AUC": roc_auc_score(y_test, rf_score),
        "PR-AUC": average_precision_score(y_test, rf_score),
    },
]).set_index("Model").round(4)
summary

## 8. Conclusion & Risk Analysis

- **Isolation Forest** (unsupervised) achieves a strong ROC-AUC (~0.94-0.95) at separating fraud from
  normal transactions using *no label information at all* — valuable in real-world settings where
  confirmed fraud labels are scarce or delayed. Its precision/recall are modest, meaning it produces
  more false positives per fraud caught.
- **Random Forest** (supervised), trained with true labels, achieves much higher precision and recall,
  showing the value of labeled data when it's available.
- **Practical takeaway:** In production, a hybrid approach is common — an unsupervised model like
  Isolation Forest flags unusual behavior in real time (including novel fraud patterns it's never
  seen before), while a supervised model, retrained periodically on confirmed cases, handles
  higher-confidence scoring for known fraud patterns.
- **Risk analysis:** the confusion matrices above show the operational trade-off directly — every
  false negative is a missed fraud (financial loss), and every false positive is a legitimate
  transaction incorrectly flagged (customer friction). The right operating threshold depends on
  which cost matters more to the business.
